In [1]:
from nsnet2_denoiser import NSnet2Enhancer

from torch_stoi import NegSTOILoss
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality
from torchmetrics.audio import SpeechReverberationModulationEnergyRatio, ShortTimeObjectiveIntelligibility, DeepNoiseSuppressionMeanOpinionScore
from torchaudio.transforms import Resample

import torch
import torchaudio
import numpy as np

from src.dataset import *

import os

In [2]:
SEED = 1984

np.random.seed(SEED)
torch.manual_seed(SEED)

gen = torch.Generator()
gen.manual_seed(SEED)

np.set_printoptions(precision=3)
torch.set_printoptions(precision=3)

In [3]:
import yaml

from NISQA_s.src.core.model_torch import model_init
from NISQA_s.src.utils.process_utils import process

NISQA_PATH = "NISQA_s/config/nisqa_s.yaml"

with open(NISQA_PATH, 'r') as stream:
    nisqa_args = yaml.safe_load(stream)
nisqa_args["ms_n_fft"] = 512
nisqa_args["hop_length"] = 256
nisqa_args["ms_win_length"] = 512
nisqa_args["ckp"] = nisqa_args["ckp"][3:]

nisqa, h0_nisqa, c0_nisqa = model_init(nisqa_args)

/home/zakhar/miniconda3/envs/ems_dereverb/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=1 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [4]:
SR = 48_000

NOISE_PATH = "data/universe-validation_set-100/input"
CLEAN_PATH = "data/universe-validation_set-100/target"
CHKP_DIR = "checkpoints"

noise_paths = [os.path.join(NOISE_PATH, x) for x in os.listdir(NOISE_PATH)]
clean_paths = [os.path.join(CLEAN_PATH, x) for x in os.listdir(CLEAN_PATH)]

test_data = list(zip(noise_paths, clean_paths))

BATCH_SIZE = 32
DEVICE = "cuda:0"

In [5]:
srmr = SpeechReverberationModulationEnergyRatio(fs=16_000, norm=False)
stoi = NegSTOILoss(16_000, use_vad=False, do_resample=False).to(DEVICE)
pesq = PerceptualEvaluationSpeechQuality(fs=16_000, mode="wb").to(DEVICE)
dnsmos = DeepNoiseSuppressionMeanOpinionScore(16_000, False, device=DEVICE)

In [6]:
from src.fspen_configs import *
from models.fspen import *

configs = TrainConfig_48khz() # TrainConfig_explicit_unfold()
# print(sum(configs.bands_num_in_groups), configs.dual_path_extension["num_modules"])
fspen = FullSubPathExtension(configs=configs)# .to(DEVICE)

state_d = torch.load(os.path.join(CHKP_DIR, "fspen_chkp", "TrainConfig_48khz_baseline#0.pt"), map_location="cpu",  weights_only=False)
fspen.eval()

FullSubPathExtension(
  (full_band_encoder): FullBandEncoder(
    (full_band_encoder): ModuleList(
      (0): FullBandEncoderBlock(
        (conv): Conv1d(2, 4, kernel_size=(6,), stride=(2,), padding=(2,))
        (norm): BatchNorm1d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
      (1): FullBandEncoderBlock(
        (conv): Conv1d(4, 16, kernel_size=(8,), stride=(2,), padding=(3,))
        (norm): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
      (2): FullBandEncoderBlock(
        (conv): Conv1d(16, 32, kernel_size=(6,), stride=(2,), padding=(2,))
        (norm): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
    )
    (global_features): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
  )
  (sub_band_encoder): SubBandEncoder_baseline(
    (sub_band_encoders): ModuleList

In [7]:
N_FFTS = configs.n_fft
HOP_LENGTH = configs.hop_length
HID_SIZE = 64
SR = configs.sample_rate

In [8]:
def vorbis_window(winlen, device="cuda"):
    sq = torch.sin(torch.pi/2*(torch.sin(torch.pi/winlen*(torch.arange(winlen)-0.5))**2)).float()
    return sq

In [9]:
window = vorbis_window(N_FFTS).to(DEVICE)
fspen = fspen.to(DEVICE)

In [10]:
noise_paths[0].split('/')[-1][:-11]

'0050b'

In [11]:
OUTPUT_PATH = "data/universe-validation_set-100/enhanced/"

In [12]:
from src.utils import model_eval, model_eval_old
from tqdm import tqdm
from scipy.io.wavfile import write

# write(AUDIO_PATH[:-4] + "_unfold.wav", SR, out_wave.cpu().detach().numpy())

def get_metrics(data, device="cpu"):
    nisqa_scores = []
    pesq_scores = []
    stoi_scores = []
    srmr_scores = []
    dnsmos_scores = []

    nisqa_scores_input = []
    pesq_scores_input = []
    stoi_scores_input = []
    srmr_scores_input = []
    dnsmos_scores_input = []

    nisqa_scores_target = []
    pesq_scores_target = []
    stoi_scores_target = []
    srmr_scores_target = []
    dnsmos_scores_target = []

    with torch.no_grad():
        for input_path, target_path in tqdm(data):
            
            signal, signal_sr = torchaudio.load(input_path)
            target, target_sr = torchaudio.load(target_path)

            signal, _ = SignalDataset.normalize_audio(signal)
            target, _ = SignalDataset.normalize_audio(target)
            # signal = signal.to(device)
            target = target.to(device)
            
            resampler = Resample(signal_sr, SR)
            sig_resampled = resampler(signal).to(device)

            spec = torch.stft(
                sig_resampled,
                n_fft=N_FFTS,
                hop_length=HOP_LENGTH,
                # onesided=True,
                win_length=N_FFTS,
                window=window,
                return_complex=True,
                normalized=True,
                center=True
            ) 

            output, _ = model_eval_old(fspen, spec, configs, device, hid_size=HID_SIZE)

            output = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                       window=window,
                       # onesided=True,
                       return_complex=False,
                       normalized=True,
                       center=True)
            
            resampler = Resample(SR, signal_sr)
            output = resampler(output.cpu()).to(device)

            output = output / (output.abs().max() / signal.abs().max())

            write(OUTPUT_PATH + input_path.split('/')[-1][:-11] + "_enhanced.wav", signal_sr, output[0].cpu().detach().numpy())

            min_l = min(output.shape[-1], target.shape[-1])

            nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
            nisqa_score_input, _, _ = process(signal.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
            nisqa_score_target, _, _ = process(target.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)

            stoi_score = stoi(output[..., :min_l], target[..., :min_l])
            signal = signal.to(device)
            stoi_score_input = stoi(signal[..., :min_l], target[..., :min_l])
            # srmr_score = srmr(output.detach().cpu())
            
            # resampler = Resample(SR, 16_000)
            # output = resampler(output.cpu()).cuda()
            # target = resampler(target.cpu()).cuda()
            
            dnsmos_score = dnsmos(output.detach())
            dnsmos_score_input = dnsmos(signal.detach())
            dnsmos_score_target = dnsmos(target.detach())
            
            min_l = min(output.shape[-1], target.shape[-1])

            srmr_score = srmr(output.detach().cpu())
            srmr_score_input = srmr(signal.detach().cpu())
            srmr_score_target = srmr(target.detach().cpu())

            try:
                pesq_score = pesq(output[..., :min_l], target[..., :min_l])
                pesq_score_input = pesq(signal[..., :min_l], target[..., :min_l])
            except Exception as e:
                # print(min_l)
                # out_wave_ = output.reshape(-1)
                # target_ = target.reshape(-1)
                # write('exception_out.wav', SR, out_wave_.cpu().detach().numpy())
                # write('exception_in.wav', SR, target_.cpu().detach().numpy())
                continue

            nisqa_scores.append(nisqa_score[0])
            srmr_scores.append(srmr_score)
            stoi_scores.append(stoi_score.cpu())
            pesq_scores.append(pesq_score.cpu())
            dnsmos_scores.append(dnsmos_score.cpu()[0])

            nisqa_scores_input.append(nisqa_score_input[0])
            srmr_scores_input.append(srmr_score_input)
            stoi_scores_input.append(stoi_score_input.cpu())
            pesq_scores_input.append(pesq_score_input.cpu())
            dnsmos_scores_input.append(dnsmos_score_input.cpu()[0])

            nisqa_scores_target.append(nisqa_score_target[0])
            srmr_scores_target.append(srmr_score_target)
            stoi_scores_target.append(1.0)
            pesq_scores_target.append(1.0)
            dnsmos_scores_target.append(dnsmos_score_target.cpu()[0])

    nisqa_scores = torch.vstack(nisqa_scores).mean(dim=0)
    stoi_scores = torch.vstack(stoi_scores).mean(dim=0)
    srmr_scores = torch.vstack(srmr_scores).mean(dim=0)
    pesq_scores = torch.vstack(pesq_scores).mean(dim=0)
    dnsmos_scores = torch.vstack(dnsmos_scores).mean(dim=0)

    nisqa_scores_input = torch.vstack(nisqa_scores_input).mean(dim=0)
    stoi_scores_input = torch.vstack(stoi_scores_input).mean(dim=0)
    srmr_scores_input = torch.vstack(srmr_scores_input).mean(dim=0)
    pesq_scores_input = torch.vstack(pesq_scores_input).mean(dim=0)
    dnsmos_scores_input = torch.vstack(dnsmos_scores_input).mean(dim=0)

    nisqa_scores_target = torch.vstack(nisqa_scores_target).mean(dim=0)
    stoi_scores_target = 1.0 # torch.vstack(stoi_scores_target).mean(dim=0)
    srmr_scores_target = torch.vstack(srmr_scores_target).mean(dim=0)
    pesq_scores_target = 4.5# torch.vstack(pesq_scores_target).mean(dim=0)
    dnsmos_scores_target = torch.vstack(dnsmos_scores_target).mean(dim=0)

    result = {"nisqa": nisqa_scores, "stoi": stoi_scores, "srmr": srmr_scores, "pesq": pesq_scores, "dnsmos": dnsmos_scores}
    result_input = {"nisqa": nisqa_scores_input, "stoi": stoi_scores_input, "srmr": srmr_scores_input, "pesq": pesq_scores_input, "dnsmos": dnsmos_scores_input}
    result_target = {"nisqa": nisqa_scores_target, "stoi": stoi_scores_target, "srmr": srmr_scores_target, "pesq": pesq_scores_target, "dnsmos": dnsmos_scores_target}

    return result, result_input, result_target

In [13]:
metrics, metrics_input, metrics_target = get_metrics(test_data, device=DEVICE)

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [04:41<00:00,  2.81s/it]


NISQA score (MOS, NOI, DISC, COL, LOUD): tensor([1.458, 2.266, 2.365, 2.413, 2.636])
STOI score: tensor([0.333])
SRMR score: tensor([2.457])
PESQ-WB score: tensor([1.109])
DNSMOS score: tensor([2.730], dtype=torch.float64)

In [14]:
print("NISQA score (MOS, NOI, DISC, COL, LOUD):", metrics["nisqa"])
print(f"STOI score: {-metrics['stoi']}")
print(f"SRMR score: {metrics['srmr']}")
print(f"PESQ-WB score: {metrics['pesq']}")
print(f"DNSMOS score: {metrics['dnsmos']}")

NISQA score (MOS, NOI, DISC, COL, LOUD): tensor([1.841, 2.483, 2.790, 2.760, 2.923])
STOI score: tensor([0.332])
SRMR score: tensor([4.123])
PESQ-WB score: tensor([1.158])
DNSMOS score: tensor([2.670], dtype=torch.float64)


In [15]:
print("NISQA score (MOS, NOI, DISC, COL, LOUD):", metrics_input["nisqa"])
print(f"STOI score: {-metrics_input['stoi']}")
print(f"SRMR score: {metrics_input['srmr']}")
print(f"PESQ-WB score: {metrics_input['pesq']}")
print(f"DNSMOS score: {metrics_input['dnsmos']}")

NISQA score (MOS, NOI, DISC, COL, LOUD): tensor([1.520, 2.138, 2.686, 2.277, 2.524])
STOI score: tensor([0.341])
SRMR score: tensor([6.063])
PESQ-WB score: tensor([1.127])
DNSMOS score: tensor([3.012], dtype=torch.float64)


In [16]:
print("NISQA score (MOS, NOI, DISC, COL, LOUD):", metrics_target["nisqa"])
print(f"STOI score: {-metrics_target['stoi']}")
print(f"SRMR score: {metrics_target['srmr']}")
print(f"PESQ-WB score: {metrics_target['pesq']}")
print(f"DNSMOS score: {metrics_target['dnsmos']}")

NISQA score (MOS, NOI, DISC, COL, LOUD): tensor([1.532, 2.808, 2.615, 1.849, 2.614])
STOI score: -1.0
SRMR score: tensor([9.927])
PESQ-WB score: 4.5
DNSMOS score: tensor([3.822], dtype=torch.float64)
